# 🗄️ NL2SQL Model — Google Colab Notebook
## Retail Analytics Copilot — `LOCAL_MODELS=False`

هذا الـ notebook يشغّل موديل **NL2SQL** على Google Colab ويكشفه عبر Ngrok.

**دور الـ NL2SQL:** يحوّل السؤال الطبيعي إلى استعلام SQL صالح لقاعدة بيانات Northwind SQLite.
- يستقبل: السؤال + schema قاعدة البيانات + القيود المستخرجة
- يُنتج: استعلام SQL جاهز للتنفيذ

**الموديل المستخدم:** `gemma2:9b-instruct-q5_0` (أكبر وأكثر دقة لتوليد SQL)

---
### ⚠️ تعليمات ما قبل التشغيل:
1. تأكد من تفعيل **GPU Runtime** → Runtime > Change runtime type > T4 GPU
2. ضع `NGROK_AUTH_TOKEN` في Colab Secrets
3. بعد التشغيل، انسخ رابط Ngrok وضعه في `.env` → `NGROK_NL2SQL_URL`

In [ ]:
# ─── الخلية 1: إيقاف أي Ollama قديم وتحديث النظام ───────────────────────────
# نوقف عمليات Ollama القديمة ونثبت zstd المطلوبة لفك ضغط نماذج Ollama
!pkill ollama || true
!apt-get update -qq
!apt-get install -y -qq zstd

In [ ]:
# ─── الخلية 2: تثبيت Ollama ────────────────────────────────────────────────
# تحميل وتثبيت Ollama runtime من الموقع الرسمي
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# ─── الخلية 3: تشغيل خادم Ollama على المنفذ 8000 ─────────────────────────────
# نشغّل الخادم على المنفذ 8000 في الخلفية
# OLLAMA_ORIGIN=* ضروري لقبول طلبات Ngrok من أي مصدر
import os
os.environ["OLLAMA_HOST"] = "127.0.0.1:8000"

!nohup bash -c "OLLAMA_HOST=127.0.0.1:8000 OLLAMA_ORIGIN=* ollama serve" > ollama_nl2sql.log 2>&1 &

# انتظر حتى يبدأ الخادم
!sleep 5

# فحص اللوجز للتأكد من نجاح البدء
!tail -20 ollama_nl2sql.log

In [ ]:
# ─── الخلية 4: تحميل موديل NL2SQL ───────────────────────────────────────────
# gemma2:9b-instruct-q5_0 = موديل Gemma 2 بحجم 9B parameters
# q5_0 = تكميم 5-bit (جودة عالية، حجم ≈6.4GB)
# هذا الموديل أكبر وأكثر دقة من phi3.5 لتوليد SQL معقد
# ⚠️ سيستغرق وقتاً أطول في التحميل (5-10 دقائق)
ollama_model_id = "gemma2:9b-instruct-q5_0"

!OLLAMA_HOST=127.0.0.1:8000 ollama pull {ollama_model_id}

# التأكد من تحميل الموديل بنجاح
!curl -s http://127.0.0.1:8000/api/tags | python3 -c "import json,sys; data=json.load(sys.stdin); [print('✓', m['name']) for m in data.get('models',[])]" 

In [ ]:
# ─── الخلية 5: اختبار توليد SQL ──────────────────────────────────────────────
# نختبر قدرة الموديل على تحويل سؤال طبيعي إلى SQL
# هذا مثال قريب من الاستخدام الحقيقي في الكوبايلوت
%%bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "gemma2:9b-instruct-q5_0",
    "messages": [
      {"role": "user", "content": "Write SQL only (no explanation): Top 3 products by revenue from Order Details table. Revenue = SUM(UnitPrice * Quantity * (1 - Discount))"}
    ],
    "stream": false
  }' | python3 -c "import json,sys; r=json.load(sys.stdin); print('SQL:', r['choices'][0]['message']['content'])"

In [ ]:
# ─── الخلية 6: تثبيت pyngrok ─────────────────────────────────────────────────
# مكتبة Python للتحكم في Ngrok tunnels
!pip install pyngrok -q

In [ ]:
# ─── الخلية 7: تشغيل Ngrok Tunnel ────────────────────────────────────────────
# نقرأ NGROK_AUTH_TOKEN من Colab Secrets
# نفتح tunnel HTTPS يربط المنفذ 8000 بالإنترنت
from google.colab import userdata
from pyngrok import ngrok, conf

# قراءة token المصادقة
ngrok_auth = userdata.get('NGROK_AUTH_TOKEN')
conf.get_default().auth_token = ngrok_auth

# إغلاق tunnels قديمة
ngrok.kill()

# فتح tunnel جديد مع ضبط host_header لـ Ollama
tunnel = ngrok.connect(
    8000,
    bind_tls=True,
    host_header="127.0.0.1:8000"
)

print("=" * 60)
print("✅ NL2SQL Model جاهز!")
print(f"🌐 NGROK_NL2SQL_URL = {tunnel.public_url}")
print("📋 انسخ هذا الرابط وضعه في ملف .env")
print("=" * 60)

In [ ]:
# ─── الخلية 8: اختبار Ngrok بسؤال SQL حقيقي ──────────────────────────────────
# نتحقق من أن الرابط العام يستجيب بشكل صحيح
import requests

ngrok_url = tunnel.public_url
model_id = ollama_model_id

resp = requests.post(
    f"{ngrok_url}/v1/chat/completions",
    json={
        "model": model_id,
        "messages": [
            {"role": "user", "content": "SQL only: What is the average order value in December 1997? Use Orders and Order Details tables."}
        ],
        "stream": False
    },
    timeout=60
)

print(f"Status: {resp.status_code}")
if resp.ok:
    answer = resp.json()["choices"][0]["message"]["content"]
    print(f"✅ Generated SQL:\n{answer}")
else:
    print(f"❌ Error: {resp.text}")